# 固定方案的测试集评估

在查看测试结果之前固定主方案、对照方案、文件指纹和评价规则，见 configs/final_test_protocol.json。所有模型使用相同的清洗后测试集；不根据测试结果调整模型或权重。


In [ ]:
# 核对固定协议及所有输入文件
import hashlib
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from sklearn.svm import LinearSVC
from sklearn.metrics import confusion_matrix

from sentiment_analysis.evaluation import classification_metrics
from sentiment_analysis.lstm import LSTMClassifier, BidirectionalLSTMClassifier
from sentiment_analysis.paths import ROOT, TABLES_DIR
from sentiment_analysis.sequence_data import SentimentDataset, TOKEN_PATTERN
from sentiment_analysis.textcnn import TextCNN

protocol_path = ROOT / "configs" / "final_test_protocol.json"
protocol_hash = hashlib.sha256(protocol_path.read_bytes()).hexdigest()
expected_protocol_hash = "ae5c7db09e1e300f700ef9b8648d378ec6392496df5ab14d932020d9b93bb876"
if protocol_hash != expected_protocol_hash:
    raise ValueError("Fixed protocol has changed; stop before test evaluation.")
protocol = json.loads(protocol_path.read_text(encoding="utf-8"))
for name, record in protocol["files"].items():
    path = ROOT / record["path"]
    if hashlib.sha256(path.read_bytes()).hexdigest() != record["sha256"]:
        raise ValueError(f"Fixed input changed: {name}")
results_path = TABLES_DIR / "fixed_models_test_results.csv"
predictions_path = TABLES_DIR / "fixed_models_test_predictions.csv"
confusion_path = TABLES_DIR / "fixed_models_test_confusion_matrices.csv"
if any(path.exists() for path in [results_path, predictions_path, confusion_path]):
    raise FileExistsError("Test result files already exist; inspect them before any rerun.")
print("Protocol and input hashes verified.")


## 训练集和验证集核对

B06 使用已保存的 TF-IDF 转换器和固定参数，在训练集重新拟合。先确认其验证集结果与之前记录一致；不在测试集拟合或调整任何模型。


In [ ]:
# 恢复 B06 并核对验证结果
train_data = pd.read_csv(ROOT / protocol["files"]["train"]["path"])
validation_data = pd.read_csv(ROOT / protocol["files"]["validation"]["path"])
vectorizer = joblib.load(ROOT / protocol["files"]["tfidf"]["path"])
X_train = vectorizer.transform(train_data["text"])
X_validation = vectorizer.transform(validation_data["text"])
svm_model = LinearSVC(C=1.0, max_iter=1000, random_state=42)
svm_model.fit(X_train, train_data["label"])
validation_prediction = svm_model.predict(X_validation)
validation_metrics = classification_metrics(validation_data["label"], validation_prediction)
baseline_record = pd.read_csv(ROOT / protocol["files"]["baseline_validation"]["path"])
baseline_record = baseline_record[baseline_record["EXPERIMENT"] == "B06"].iloc[0]
for metric, value in validation_metrics.items():
    if not np.isclose(value, baseline_record[metric], rtol=0, atol=1e-12):
        raise ValueError(f"B06 validation metric does not match: {metric}")
print("B06 validation results verified.")


In [ ]:
# 加载固定神经模型，核对词表和序列处理规则
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoints = {
    name: torch.load(ROOT / protocol["files"][name]["path"], map_location=device, weights_only=True)
    for name in ["textcnn", "lstm", "bilstm"]
}
reference = checkpoints["textcnn"]
for name, checkpoint in checkpoints.items():
    if (checkpoint["vocabulary"] != reference["vocabulary"]
            or checkpoint["max_length"] != reference["max_length"]
            or checkpoint["token_pattern"] != TOKEN_PATTERN.pattern
            or not checkpoint["lowercase"]
            or checkpoint["random_seed"] != 42):
        raise ValueError(f"Checkpoint processing or seed mismatch: {name}")

models = {
    "TextCNN": TextCNN(**checkpoints["textcnn"]["model_config"]).to(device),
    "LSTM": LSTMClassifier(**checkpoints["lstm"]["model_config"]).to(device),
    "BiLSTM": BidirectionalLSTMClassifier(**checkpoints["bilstm"]["model_config"]).to(device),
}
for model_name, key in [("TextCNN","textcnn"),("LSTM","lstm"),("BiLSTM","bilstm")]:
    models[model_name].load_state_dict(checkpoints[key]["model_state_dict"])
    models[model_name].eval()
print("Neural model checkpoints and vocabulary verified.")


## 测试集评估

使用预先固定的检查点和 0.5 / 0.5 权重，对相同测试样本生成预测和评价指标。此处不搜索参数或阈值。


In [ ]:
# 检查清洗后数据集，并准备测试序列
test_data = pd.read_csv(ROOT / protocol["files"]["test"]["path"])
for name, data in [("train", train_data), ("validation", validation_data), ("test", test_data)]:
    if data["text"].isna().any() or data["text"].duplicated().any() or not data["label"].isin([0,1]).all():
        raise ValueError(f"Invalid text, labels, or duplicate texts in {name}.")
if (set(train_data["text"]) & set(validation_data["text"])
        or set(train_data["text"]) & set(test_data["text"])
        or set(validation_data["text"]) & set(test_data["text"])):
    raise ValueError("Overlapping exact texts found between splits.")
test_dataset = SentimentDataset(test_data, reference["vocabulary"], reference["max_length"])
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
print(f"Test samples: {len(test_dataset):,}")


In [ ]:
# 为三个固定神经模型生成正类概率
labels_all = []
probabilities = {name: [] for name in models}
with torch.no_grad():
    for token_ids, lengths, labels in test_loader:
        token_ids = token_ids.to(device)
        labels_all.extend(labels.tolist())
        for name, model in models.items():
            logits = model(token_ids, lengths)
            probabilities[name].extend(torch.softmax(logits, dim=1)[:, 1].cpu().tolist())

if labels_all != test_data["label"].tolist():
    raise ValueError("Test labels are not aligned.")
for name in probabilities:
    probabilities[name] = np.array(probabilities[name])
    if not np.isfinite(probabilities[name]).all():
        raise ValueError(f"Non-finite probabilities: {name}")

cnn_probability = probabilities["TextCNN"]
lstm_probability = probabilities["LSTM"]
bilstm_probability = probabilities["BiLSTM"]
predictions = {
    "B06": svm_model.predict(vectorizer.transform(test_data["text"])),
    "TextCNN": (cnn_probability > 0.5).astype(int),
    "LSTM": (lstm_probability > 0.5).astype(int),
    "BiLSTM": (bilstm_probability > 0.5).astype(int),
    "TextCNN + LSTM (equal)": ((cnn_probability + lstm_probability) / 2 > 0.5).astype(int),
    "TextCNN + BiLSTM (equal)": ((cnn_probability + bilstm_probability) / 2 > 0.5).astype(int),
}
print(f"Predictions per model: {len(labels_all):,}")


In [ ]:
# 计算固定方案的测试指标和混淆矩阵
results = []
confusion_rows = []
for name, prediction in predictions.items():
    if len(prediction) != len(labels_all):
        raise ValueError(f"Prediction count mismatch: {name}")
    results.append({"MODEL": name, **classification_metrics(labels_all, prediction)})
    matrix = confusion_matrix(labels_all, prediction, labels=[0, 1])
    confusion_rows.append({
        "MODEL": name, "TN": matrix[0,0], "FP": matrix[0,1],
        "FN": matrix[1,0], "TP": matrix[1,1],
    })

test_results = pd.DataFrame(results)
test_confusion = pd.DataFrame(confusion_rows)
display(test_results.round(4))
display(test_confusion)


In [ ]:
# 保存完整指标、混淆矩阵和逐样本预测
TABLES_DIR.mkdir(parents=True, exist_ok=True)
prediction_table = pd.DataFrame({
    "TEST ROW": np.arange(len(labels_all)),
    "LABEL": labels_all,
    "TEXTCNN POSITIVE PROBABILITY": cnn_probability,
    "LSTM POSITIVE PROBABILITY": lstm_probability,
    "BILSTM POSITIVE PROBABILITY": bilstm_probability,
    **predictions,
})
test_results.to_csv(results_path, index=False)
test_confusion.to_csv(confusion_path, index=False)
prediction_table.to_csv(predictions_path, index=False)
print(f"Protocol SHA-256: {protocol_hash}")
print(f"Saved: {results_path}, {confusion_path}, {predictions_path}")
